# Selective Attention Benchmark

By Conrad Kleykamp

This benchmark was completed as part of the Kaggle and Google DeepMind [Measuring Progress Towards AGI - Cognitive Abilities](https://www.kaggle.com/competitions/kaggle-measuring-agi/overview) hackathon. The overall submission for this hackathon includes: Selective Attention, Sustained Attention, and Divided Attention.

Evaluates whether models can extract goal-relevant information from passages containing embedded distractors. Runs across all models available in the Kaggle environment (`kbench.llms`). A fixed judge model (`kbench.judge_llm`) scores all responses to ensure consistent cross-model evaluation.

Each task row contains 5 judge-evaluated criteria. Total assertions per model: 15.

> **Environment note:** Designed to run inside a Kaggle notebook where `kbench.llm`, `kbench.judge_llm`, and `kbench.llms` are auto-configured.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

In [ ]:
TASK_DATA = pd.DataFrame([
    {
        "context": (
            "The annual report stated that revenue grew by 12% in Q3. "
            "The marketing team launched three new campaigns in August. "
            "Operating costs fell by 4% due to supply chain improvements. "
            "The CEO mentioned plans to expand into Asian markets next year. "
            "Employee headcount increased by 200 in the past quarter. "
            "Net profit margin improved from 8.1% to 9.6% year-over-year."
        ),
        "question": "What financial performance metrics are reported?",
        "distractors": ["marketing campaigns", "Asian market expansion", "headcount"],
        "criteria": [
            "Response includes the 12% revenue growth in Q3.",
            "Response includes the 4% reduction in operating costs.",
            "Response includes the improvement in net profit margin from 8.1% to 9.6%.",
            "Response does not reference marketing campaigns or headcount as financial metrics.",
            "Response does not cite the planned Asian market expansion as a financial performance metric.",
        ],
    },
    {
        "context": (
            "The patient presented with a fever of 101.3F and mild cough. "
            "She mentioned her dog recently had surgery for a broken leg. "
            "Blood pressure was recorded at 118/76 mmHg. "
            "Her teenage son started college last month. "
            "Oxygen saturation measured at 97%. "
            "The patient reported occasional headaches over the past week. "
            "Her neighbor recommended a new restaurant downtown."
        ),
        "question": "What are the patient's clinical observations?",
        "distractors": ["dog surgery", "son starting college", "restaurant recommendation"],
        "criteria": [
            "Response includes the fever of 101.3F.",
            "Response includes the blood pressure reading of 118/76 mmHg.",
            "Response includes oxygen saturation of 97%.",
            "Response includes the mild cough and occasional headaches.",
            "Response excludes irrelevant personal details about the dog, son, or restaurant.",
        ],
    },
    {
        "context": (
            "The experiment used a sample size of 342 participants aged 18-65. "
            "The lead researcher enjoys hiking on weekends. "
            "Treatment group showed a 23% reduction in symptoms after 8 weeks. "
            "Data collection took place across four university hospitals. "
            "The control group received a placebo with no significant changes observed. "
            "Funding was provided by a private health foundation. "
            "Three participants withdrew due to personal reasons unrelated to the study."
        ),
        "question": "What were the key experimental findings and methodology?",
        "distractors": ["researcher hobbies", "funding source", "withdrawal reasons"],
        "criteria": [
            "Response includes the sample size of 342 participants.",
            "Response includes the 23% symptom reduction in the treatment group.",
            "Response includes the placebo control group outcome.",
            "Response includes the 8-week treatment duration.",
            "Response does not dwell on the researcher's hobbies or unrelated withdrawal reasons.",
        ],
    },
])


@kbench.task(name="selective_attention")
def selective_attention(
    llm,
    context: str,
    question: str,
    distractors: list,
    criteria: list,
):
    """
    Evaluating selective attention: model must extract goal-relevant information
    from a passage containing embedded distractors. Judge is a fixed external model.
    """
    prompt = (
        f"Read the following passage carefully and answer the question. "
        f"Focus only on information directly relevant to the question.\n\n"
        f"Passage:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    report = kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=kbench.judge_llm,
    )
    for result in report.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=result.criterion,
        )

In [ ]:
all_runs = []

for model_name, model_llm in kbench.llms.items():
    print(f"\n--- {model_name} ---")
    for i, row in TASK_DATA.iterrows():
        run = selective_attention.run(
            llm=model_llm,
            context=row["context"],
            question=row["question"],
            distractors=row["distractors"],
            criteria=row["criteria"],
        )
        all_runs.append((model_name, i, run))
        passed = sum(ar.passed for ar in run.assertion_results)
        total = len(run.assertion_results)
        print(f"  Row {i}: {run.status.name} -- {passed}/{total} assertions passed")

In [ ]:
def run_to_record(model_name, row_index, run):
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    assertion_summary = "; ".join(
        f"{'PASS' if ar.passed else 'FAIL'}: {ar.expectation}"
        for ar in run.assertion_results
    )
    return {
        "model": model_name,
        "row": row_index,
        "status": run.status.name,
        "assertions_passed": passed,
        "assertions_total": total,
        "pass_rate": round(passed / total, 2) if total > 0 else None,
        "error_message": run.error_message,
        "assertion_details": assertion_summary,
    }


records = [run_to_record(model_name, i, run) for model_name, i, run in all_runs]
results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 100)

print("\n=== Per-row results ===")
display(results_df)

print("\n=== Summary by model ===")
summary = (
    results_df.groupby("model")
    .agg(
        assertions_passed=("assertions_passed", "sum"),
        assertions_total=("assertions_total", "sum"),
    )
    .assign(pass_rate=lambda df: (df["assertions_passed"] / df["assertions_total"]).round(3))
    .sort_values("pass_rate", ascending=False)
    .reset_index()
)
display(summary)

In [ ]:
output_path = "selective_attention_results.csv"
summary.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
display(summary)